# Failure Analysis: Copilot Opus 4.5

Analysis of manually-reviewed failure categories across 5 independent runs (101 tasks x 5 runs = 505 result entries).

**Failure Categories** (from `reviewer.py`):
- **Wrong Solution**: Incorrect approach/logic
- **Syntax Error**: AL syntax issues
- **Incorrect File**: Wrong file edited
- **Missing Using**: Missing using statement for namespace
- **Timeout**: Agent timed out and made no changes
- **Other**: Doesn't fit above

In [1]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from utils import get_result_folder, load_results_df

FAILURE_CATEGORIES = ["Wrong Solution", "Syntax Error", "Incorrect File", "Missing Using", "Timeout", "Other"]

In [2]:
# Load all results from the 5 runs
setup_folder = get_result_folder("bug-fix") / "copilot-opus-4-5"
df = load_results_df(setup_folder)

# Add alias for backwards compat
df["builds"] = df["build"]

print(f"Loaded {len(df)} results from {df['run_id'].nunique()} runs")
print(f"Unique tasks: {df['instance_id'].nunique()}")
df.head()

Loaded 505 results from 5 runs
Unique tasks: 101


,run_id,instance_id,project,resolved,build,execution_time,llm_duration,turn_count,total_tokens,tool_calls,tool_usage,failure_category,output,builds
0,21441655230,microsoftInternal__NAV-218062,BaseApp,False,False,512.775,426.239,59.0,2119600,58,"{'edit': 1, 'glob': 1, 'grep': 26, 'report_int...",Missing Using,diff --git a/App/Layers/W1/BaseApp/Warehouse/A...,False
1,21441655230,microsoftInternal__NAV-193649,Shopify,False,False,121.149,106.288,22.0,810900,31,"{'create': 1, 'grep': 17, 'report_intent': 2, ...",Syntax Error,diff --git a/App/Apps/W1/Shopify/app/src/Order...,False
2,21441655230,microsoftInternal__NAV-224447,BaseApp,False,False,222.267,127.056,38.0,1106100,30,"{'edit': 2, 'grep': 13, 'report_intent': 2, 't...",Missing Using,diff --git a/App/Layers/W1/BaseApp/Foundation/...,False
3,21441655230,microsoftInternal__NAV-178045,BaseApp,False,False,1800.000,0.000,NaN,0,0,None,Timeout,,False
4,21441655230,microsoftInternal__NAV-191624,ExcelReports,False,False,39.000,30.000,6.0,81000,8,"{'edit': 2, 'grep': 1, 'report_intent': 2, 'vi...",Syntax Error,diff --git a/App/Apps/W1/ExcelReports/app/src/...,False


## 1. Overview Statistics

In [3]:
# Overall success/failure breakdown
total = len(df)
resolved = df["resolved"].sum()
failed = total - resolved
builds_ok = df["builds"].sum()
build_failed = total - builds_ok

print(f"Total entries: {total}")
print(f"Resolved: {resolved} ({resolved / total * 100:.1f}%)")
print(f"Failed: {failed} ({failed / total * 100:.1f}%)")
print(f"Build succeeded: {builds_ok} ({builds_ok / total * 100:.1f}%)")
print(f"Build failed: {build_failed} ({build_failed / total * 100:.1f}%)")

Total entries: 505
Resolved: 295 (58.4%)
Failed: 210 (41.6%)
Build succeeded: 484 (95.8%)
Build failed: 21 (4.2%)


In [4]:
# Filter to failures only for category analysis
failures_df = df[~df["resolved"]].copy()
print(f"\nAnalyzing {len(failures_df)} failed entries")

# Check review coverage
reviewed = failures_df["failure_category"].notna().sum()
unreviewed = len(failures_df) - reviewed
print(f"Reviewed: {reviewed} ({reviewed / len(failures_df) * 100:.1f}%)")
print(f"Unreviewed: {unreviewed} ({unreviewed / len(failures_df) * 100:.1f}%)")


Analyzing 210 failed entries
Reviewed: 210 (100.0%)
Unreviewed: 0 (0.0%)


## 2. Failure Category Distribution

In [5]:
# Replace NaN with "Unreviewed" for visualization
failures_df["category"] = failures_df["failure_category"].fillna("Unreviewed")

# Count by category
category_counts = failures_df["category"].value_counts().reset_index()
category_counts.columns = ["category", "count"]
category_counts["percentage"] = (category_counts["count"] / len(failures_df) * 100).round(1)

# Define color mapping
color_map = {
    "Wrong Solution": "#EF553B",
    "Syntax Error": "#FFA15A",
    "Incorrect File": "#FECB52",
    "Missing Using": "#00CC96",
    "Timeout": "#636EFA",
    "Other": "#AB63FA",
    "Unreviewed": "#B6E880",
}

fig = px.pie(
    category_counts,
    values="count",
    names="category",
    title="Failure Category Distribution (All Runs)",
    color="category",
    color_discrete_map=color_map,
    hole=0.4,
)
fig.update_traces(textposition="outside", textinfo="percent+label+value")
fig.show()

In [6]:
# Bar chart with build success breakdown
category_build = failures_df.groupby(["category", "builds"]).size().reset_index(name="count")
category_build["build_status"] = category_build["builds"].map({True: "Build OK", False: "Build Failed"})

fig = px.bar(
    category_build,
    x="category",
    y="count",
    color="build_status",
    title="Failure Categories by Build Status",
    barmode="stack",
    color_discrete_map={"Build OK": "#00CC96", "Build Failed": "#EF553B"},
    category_orders={"category": [*FAILURE_CATEGORIES, "Unreviewed"]},
)
fig.update_layout(xaxis_title="Failure Category", yaxis_title="Count")
fig.show()

## 3. Cross-Run Consistency Analysis

Do the same tasks fail the same way across all 5 runs?

In [7]:
# Pivot: instance_id x run_id -> resolution status
resolution_pivot = df.pivot_table(index="instance_id", columns="run_id", values="resolved", aggfunc="first")

# Calculate consistency metrics
n_runs = resolution_pivot.shape[1]
resolution_pivot["pass_count"] = resolution_pivot.sum(axis=1)
resolution_pivot["fail_count"] = n_runs - resolution_pivot["pass_count"]

# Categorize tasks
always_pass = (resolution_pivot["pass_count"] == n_runs).sum()
always_fail = (resolution_pivot["fail_count"] == n_runs).sum()
flaky = n_runs - always_pass - always_fail

print(f"Tasks that ALWAYS pass (5/5 runs): {always_pass}")
print(f"Tasks that ALWAYS fail (0/5 runs): {always_fail}")
print(f"Flaky tasks (pass sometimes): {len(resolution_pivot) - always_pass - always_fail}")

Tasks that ALWAYS pass (5/5 runs): 39
Tasks that ALWAYS fail (0/5 runs): 23
Flaky tasks (pass sometimes): 39


In [8]:
# Visualize pass/fail consistency
pass_distribution = resolution_pivot["pass_count"].value_counts().sort_index().reset_index()
pass_distribution.columns = ["passes_out_of_5", "task_count"]

fig = px.bar(
    pass_distribution,
    x="passes_out_of_5",
    y="task_count",
    title="Task Resolution Consistency Across 5 Runs",
    labels={"passes_out_of_5": "# of Runs Passed", "task_count": "# of Tasks"},
    text="task_count",
    color="passes_out_of_5",
    color_continuous_scale="RdYlGn",
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [9]:
# For failures: analyze category consistency
failure_category_pivot = failures_df.pivot_table(index="instance_id", columns="run_id", values="category", aggfunc="first")


# Calculate most common category per task and consistency
def get_category_stats(row):
    categories = row.dropna().tolist()
    if not categories:
        return pd.Series({"most_common": None, "consistency": 0, "n_failures": 0})
    counter = Counter(categories)
    most_common, count = counter.most_common(1)[0]
    return pd.Series(
        {
            "most_common": most_common,
            "consistency": count / len(categories),
            "n_failures": len(categories),
            "unique_categories": len(counter),
        }
    )


category_consistency = failure_category_pivot.apply(get_category_stats, axis=1)
category_consistency = category_consistency[category_consistency["n_failures"] > 0]

print(f"\nCategory Consistency for {len(category_consistency)} tasks that failed at least once:")
print(f"Tasks with 100% consistent failure category: {(category_consistency['consistency'] == 1).sum()}")
print(f"Tasks with mixed failure categories: {(category_consistency['consistency'] < 1).sum()}")
print(f"\nAverage consistency score: {category_consistency['consistency'].mean():.1%}")


Category Consistency for 62 tasks that failed at least once:
Tasks with 100% consistent failure category: 50
Tasks with mixed failure categories: 12

Average consistency score: 93.8%


In [10]:
# Show tasks with inconsistent failure categories (fail differently across runs)
inconsistent = category_consistency[category_consistency["consistency"] < 1].copy()
if len(inconsistent) > 0:
    print(f"\nTasks with inconsistent failure categories ({len(inconsistent)} tasks):")
    inconsistent_details = inconsistent.join(failure_category_pivot)
    display(inconsistent_details.sort_values("consistency"))
else:
    print("All tasks fail consistently with the same category!")


Tasks with inconsistent failure categories (12 tasks):


,most_common,consistency,n_failures,unique_categories,21441655230,21449814776,21469225831,21475222924,21483543210
instance_id,,,,,,,,,
microsoftInternal__NAV-215645,Incorrect File,0.400000,5,3,Incorrect File,Other,Incorrect File,Syntax Error,Other
microsoftInternal__NAV-214557,Wrong Solution,0.500000,4,3,Wrong Solution,NaN,Incorrect File,Missing Using,Wrong Solution
microsoftInternal__NAV-224447,Missing Using,0.500000,2,2,Missing Using,NaN,NaN,NaN,Incorrect File
microsoftInternal__NAV-178045,Timeout,0.666667,3,2,Timeout,NaN,NaN,Timeout,Incorrect File
microsoft__BCApps-5633,Wrong Solution,0.666667,3,2,NaN,Wrong Solution,Wrong Solution,NaN,Missing Using
microsoftInternal__NAV-193649,Missing Using,0.666667,3,2,Syntax Error,Missing Using,Missing Using,NaN,NaN
microsoftInternal__NAV-218062,Missing Using,0.750000,4,2,Missing Using,NaN,Missing Using,Wrong Solution,Missing Using
microsoftInternal__NAV-209450,Incorrect File,0.800000,5,2,Incorrect File,Incorrect File,Incorrect File,Other,Incorrect File
microsoftInternal__NAV-217797,Incorrect File,0.800000,5,2,Incorrect File,Incorrect File,Incorrect File,Wrong Solution,Incorrect File


## 4. Consistently Failing Tasks

Which tasks fail in ALL 5 runs with the same category?

In [11]:
# Tasks that fail all 5 runs with same category
always_fail_tasks = resolution_pivot[resolution_pivot["fail_count"] == n_runs].index.tolist()
consistent_failures = category_consistency[(category_consistency.index.isin(always_fail_tasks)) & (category_consistency["consistency"] == 1)]

print(f"Tasks that fail ALL 5 runs with the SAME category: {len(consistent_failures)}")

# Group by category
if len(consistent_failures) > 0:
    by_category = consistent_failures.groupby("most_common").size().sort_values(ascending=False)
    print("\nBreakdown by failure category:")
    for cat, count in by_category.items():
        print(f"  {cat}: {count} tasks")

Tasks that fail ALL 5 runs with the SAME category: 17

Breakdown by failure category:
  Wrong Solution: 14 tasks
  Syntax Error: 2 tasks
  Incorrect File: 1 tasks


In [12]:
# Detailed table of consistently failing tasks
if len(consistent_failures) > 0:
    # Add project info
    task_projects = df.groupby("instance_id")["project"].first()
    consistent_failures_detail = consistent_failures.copy()
    consistent_failures_detail["project"] = consistent_failures_detail.index.map(task_projects)

    display(consistent_failures_detail[["most_common", "project"]].sort_values(["most_common", "project"]).rename(columns={"most_common": "failure_category"}))

,failure_category,project
instance_id,,
microsoftInternal__NAV-213741,Incorrect File,BaseApp
microsoftInternal__NAV-183399,Syntax Error,BaseApp
microsoftInternal__NAV-191624,Syntax Error,ExcelReports
microsoftInternal__NAV-176426,Wrong Solution,BaseApp
microsoftInternal__NAV-177750,Wrong Solution,BaseApp
microsoftInternal__NAV-181900,Wrong Solution,BaseApp
microsoftInternal__NAV-182354,Wrong Solution,BaseApp
microsoftInternal__NAV-204450,Wrong Solution,BaseApp
microsoftInternal__NAV-206135,Wrong Solution,BaseApp


## 5. Namespace Changes vs Missing Using Failures

Do tasks that require `namespace` or `using` changes in the gold patch correlate with "Missing Using" failures?

10 trials failed due to "Missing Using", could it be that namespace is relavtively new in AL so that models don't know?

In [13]:
import json


def touches_namespace(patch: str) -> bool:
    """
    Check if a patch touches namespace-related lines.
    Looks for add/delete of lines starting with 'namespace' or 'using'.
    """
    if not patch:
        return False

    for line in patch.split("\n"):
        # Skip file headers
        if line.startswith(("+++", "---")):
            continue

        # Check added or deleted lines
        if line.startswith(("+", "-")):
            content = line[1:].strip()
            if content.startswith(("namespace", "using")):
                return True
    return False


# Load bcbench.jsonl and check namespace changes
dataset_path = Path.cwd().parent.parent / "dataset" / "bcbench.jsonl"
namespace_info = {}

with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        instance_id = entry.get("instance_id", "")
        patch = entry.get("patch", "")
        test_patch = entry.get("test_patch", "")

        namespace_info[instance_id] = {
            "patch_touches_ns": touches_namespace(patch),
            "test_patch_touches_ns": touches_namespace(test_patch),
            "any_touches_ns": touches_namespace(patch) or touches_namespace(test_patch),
        }

# Add namespace info to main df
df["gold_patch_touches_namespace"] = df["instance_id"].map(lambda x: namespace_info.get(x, {}).get("patch_touches_ns", False))
df["gold_any_touches_namespace"] = df["instance_id"].map(lambda x: namespace_info.get(x, {}).get("any_touches_ns", False))

# Summary of tasks
ns_tasks = df.groupby("instance_id")["gold_patch_touches_namespace"].first()
print(f"Tasks where gold patch touches namespace/using: {ns_tasks.sum()} / {len(ns_tasks)}\n")

# Analyze resolved rate for tasks where gold patch touches namespace
ns_task_ids = ns_tasks[ns_tasks].index.tolist()
df_ns_tasks = df[df["instance_id"].isin(ns_task_ids)]
df_non_ns_tasks = df[~df["instance_id"].isin(ns_task_ids)]

print("=== Resolved Rate Comparison ===\n")
print(f"Tasks where gold patch touches namespace ({len(ns_task_ids)} tasks, {len(df_ns_tasks)} runs):")
ns_resolved = df_ns_tasks["resolved"].sum()
print(f"  Resolved: {ns_resolved} / {len(df_ns_tasks)} ({100 * ns_resolved / len(df_ns_tasks):.1f}%)")

print(f"\nTasks where gold patch does NOT touch namespace ({len(ns_tasks) - len(ns_task_ids)} tasks, {len(df_non_ns_tasks)} runs):")
non_ns_resolved = df_non_ns_tasks["resolved"].sum()
print(f"  Resolved: {non_ns_resolved} / {len(df_non_ns_tasks)} ({100 * non_ns_resolved / len(df_non_ns_tasks):.1f}%)")

# Per-task breakdown for namespace-touching tasks
print("\n=== Per-Task Breakdown (Namespace-Touching Tasks) ===\n")
ns_task_stats = (
    df_ns_tasks.groupby("instance_id")
    .agg(
        runs=("resolved", "count"),
        resolved=("resolved", "sum"),
    )
    .reset_index()
)
ns_task_stats["resolve_rate"] = (ns_task_stats["resolved"] / ns_task_stats["runs"] * 100).round(1).astype(str) + "%"
display(ns_task_stats)

Tasks where gold patch touches namespace/using: 11 / 101

=== Resolved Rate Comparison ===

Tasks where gold patch touches namespace (11 tasks, 55 runs):
  Resolved: 20 / 55 (36.4%)

Tasks where gold patch does NOT touch namespace (90 tasks, 450 runs):
  Resolved: 275 / 450 (61.1%)

=== Per-Task Breakdown (Namespace-Touching Tasks) ===



,instance_id,runs,resolved,resolve_rate
0,microsoftInternal__NAV-175577,5,5,100.0%
1,microsoftInternal__NAV-176194,5,3,60.0%
2,microsoftInternal__NAV-193649,5,2,40.0%
3,microsoftInternal__NAV-209450,5,0,0.0%
4,microsoftInternal__NAV-214557,5,1,20.0%
5,microsoftInternal__NAV-214926,5,4,80.0%
6,microsoftInternal__NAV-216057,5,0,0.0%
7,microsoftInternal__NAV-216918,5,0,0.0%
8,microsoftInternal__NAV-223790,5,1,20.0%
9,microsoftInternal__NAV-226448,5,1,20.0%


In [14]:
# Analyze failure reasons for the 11 namespace-touching tasks (55 trials total)
df_ns_tasks_all = df[df["instance_id"].isin(ns_task_ids)].copy()
df_ns_tasks_all["category"] = df_ns_tasks_all["failure_category"].fillna("Unreviewed")
df_ns_tasks_all.loc[df_ns_tasks_all["resolved"], "category"] = "Resolved"

print(f"=== Outcome Distribution for Namespace-Touching Tasks ({len(ns_task_ids)} tasks x 5 runs = {len(df_ns_tasks_all)} trials) ===\n")

# Overall outcome distribution
outcome_counts = df_ns_tasks_all["category"].value_counts()
print("Outcome breakdown:")
for outcome, count in outcome_counts.items():
    print(f"  {outcome}: {count} ({100 * count / len(df_ns_tasks_all):.1f}%)")

# Visualize
fig = px.pie(
    values=outcome_counts.values,
    names=outcome_counts.index,
    title=f"Outcomes for {len(ns_task_ids)} Tasks Where Gold Patch Touches Namespace (55 trials)",
    color=outcome_counts.index,
    color_discrete_map={
        "Resolved": "#00CC96",
        "Wrong Solution": "#EF553B",
        "Syntax Error": "#FFA15A",
        "Incorrect File": "#FECB52",
        "Missing Using": "#19D3F3",
        "Timeout": "#636EFA",
        "Other": "#AB63FA",
        "Unreviewed": "#B6E880",
    },
    hole=0.4,
)
fig.update_traces(textposition="outside", textinfo="percent+label+value")
fig.show()

=== Outcome Distribution for Namespace-Touching Tasks (11 tasks × 5 runs = 55 trials) ===

Outcome breakdown:
  Resolved: 20 (36.4%)
  Wrong Solution: 14 (25.5%)
  Incorrect File: 14 (25.5%)
  Missing Using: 5 (9.1%)
  Syntax Error: 1 (1.8%)
  Other: 1 (1.8%)


In [15]:
# Check if agent's solution touches namespace/using
df_ns_tasks_all["agent_touches_namespace"] = df_ns_tasks_all["output"].apply(touches_namespace)

print("=== Agent's Namespace Usage for Namespace-Touching Tasks ===\n")

# Compare gold vs agent namespace touching
comparison = (
    df_ns_tasks_all.groupby("instance_id")
    .agg(
        gold_touches_ns=("gold_patch_touches_namespace", "first"),
        agent_touches_ns_any=("agent_touches_namespace", "any"),  # Did agent touch ns in ANY run?
        agent_touches_ns_count=("agent_touches_namespace", "sum"),  # How many runs did agent touch ns?
        resolved_count=("resolved", "sum"),
    )
    .reset_index()
)

print("Per-task comparison (gold patch requires namespace change):\n")
display(comparison)

# Overall stats
agent_touched_ns = df_ns_tasks_all["agent_touches_namespace"].sum()
print(f"\nAgent touched namespace/using in {agent_touched_ns} / {len(df_ns_tasks_all)} trials ({100 * agent_touched_ns / len(df_ns_tasks_all):.1f}%)")

# Cross-tab: Did agent touch namespace vs outcome?
print("\n=== Outcome by Whether Agent Touched Namespace ===\n")
crosstab = df_ns_tasks_all.groupby(["agent_touches_namespace", "category"]).size().unstack(fill_value=0)
display(crosstab)

# For failures only: did agent attempt namespace changes?
ns_failures = df_ns_tasks_all[~df_ns_tasks_all["resolved"]]
if len(ns_failures) > 0:
    agent_tried_ns = ns_failures["agent_touches_namespace"].sum()
    print(f"\nAmong {len(ns_failures)} failed trials on namespace-touching tasks:")
    print(f"  Agent DID touch namespace/using: {agent_tried_ns} ({100 * agent_tried_ns / len(ns_failures):.1f}%)")
    print(f"  Agent did NOT touch namespace/using: {len(ns_failures) - agent_tried_ns} ({100 * (len(ns_failures) - agent_tried_ns) / len(ns_failures):.1f}%)")

=== Agent's Namespace Usage for Namespace-Touching Tasks ===

Per-task comparison (gold patch requires namespace change):



,instance_id,gold_touches_ns,agent_touches_ns_any,agent_touches_ns_count,resolved_count
0,microsoftInternal__NAV-175577,True,False,0,5
1,microsoftInternal__NAV-176194,True,False,0,3
2,microsoftInternal__NAV-193649,True,True,4,2
3,microsoftInternal__NAV-209450,True,False,0,0
4,microsoftInternal__NAV-214557,True,False,0,1
5,microsoftInternal__NAV-214926,True,False,0,4
6,microsoftInternal__NAV-216057,True,False,0,0
7,microsoftInternal__NAV-216918,True,False,0,0
8,microsoftInternal__NAV-223790,True,True,2,1
9,microsoftInternal__NAV-226448,True,False,0,1



Agent touched namespace/using in 9 / 55 trials (16.4%)

=== Outcome by Whether Agent Touched Namespace ===



category,Incorrect File,Missing Using,Other,Resolved,Syntax Error,Wrong Solution
agent_touches_namespace,,,,,,
False,12,4,1,15,0,14
True,2,1,0,5,1,0



Among 35 failed trials on namespace-touching tasks:
  Agent DID touch namespace/using: 4 (11.4%)
  Agent did NOT touch namespace/using: 31 (88.6%)


## 6. Localization Analysis: W1 vs Other Folders

The NAV repository contains various localizations. Our evaluation harness only verifies the agent's output in the **W1** folder (by running tests there).

Some fixes might require changes across multiple localizations, but we only evaluate W1. Does the agent touch folders beyond W1, and if so, does it affect resolution rate?

In [16]:
from utils import extract_localizations_from_patch

# Analyze agent outputs for localization touches
df["agent_localizations"] = df["output"].apply(extract_localizations_from_patch)
df["touches_w1"] = df["agent_localizations"].apply(lambda x: "W1" in x)
df["touches_non_w1"] = df["agent_localizations"].apply(lambda x: len(x - {"W1"}) > 0)
df["touches_only_w1"] = df["agent_localizations"].apply(lambda x: x == {"W1"})
df["localization_count"] = df["agent_localizations"].apply(len)

print("=== Localization Folders Touched by Agent ===\n")

# Overall stats
total_with_output = df[df["output"].notna() & (df["output"] != "")].shape[0]
touches_w1_only = df["touches_only_w1"].sum()
touches_non_w1_total = df["touches_non_w1"].sum()

print(f"Total trials with agent output: {total_with_output}")
print(f"Trials touching ONLY W1: {touches_w1_only}")
print(f"Trials touching non-W1 localizations: {touches_non_w1_total}")

# Show distribution of localization counts (exclude 0 - those are timeouts with empty output)
loc_dist = df[df["localization_count"] > 0]["localization_count"].value_counts().sort_index()
print("\nDistribution of # localizations touched:")
for count, n_trials in loc_dist.items():
    pct = 100 * n_trials / len(df)
    print(f"  {count} localizations: {n_trials} trials ({pct:.1f}%)")

=== Localization Folders Touched by Agent ===

Total trials with agent output: 503
Trials touching ONLY W1: 453
Trials touching non-W1 localizations: 50

Distribution of # localizations touched:
  1 localizations: 453 trials (89.7%)
  2 localizations: 20 trials (4.0%)
  3 localizations: 2 trials (0.4%)
  4 localizations: 3 trials (0.6%)
  5 localizations: 5 trials (1.0%)
  6 localizations: 4 trials (0.8%)
  7 localizations: 1 trials (0.2%)
  8 localizations: 6 trials (1.2%)
  9 localizations: 5 trials (1.0%)
  11 localizations: 2 trials (0.4%)
  15 localizations: 1 trials (0.2%)
  16 localizations: 1 trials (0.2%)


In [17]:
# Which non-W1 localizations are being touched?
all_localizations = set()
for locs in df["agent_localizations"]:
    all_localizations.update(locs)

print("=== Non-W1 Localizations Touched ===\n")
non_w1_locs = all_localizations - {"W1"}
if non_w1_locs:
    print(f"Non-W1 localizations: {sorted(non_w1_locs)}")

    # Count how often each localization is touched
    loc_counts = Counter()
    for locs in df["agent_localizations"]:
        for loc in locs:
            loc_counts[loc] += 1

    print("\nFrequency of each localization:")
    for loc, count in sorted(loc_counts.items(), key=lambda x: -x[1]):
        print(f"  {loc}: {count} trials")
else:
    print("No non-W1 localizations touched by agent outputs.")

=== Non-W1 Localizations Touched ===

Non-W1 localizations: ['APAC', 'AU', 'BE', 'CH', 'CZ', 'DACH', 'ES', 'FI', 'FR', 'GB', 'IT', 'NA', 'NL', 'NO', 'RU', 'SE']

Frequency of each localization:
  W1: 503 trials
  IT: 39 trials
  APAC: 25 trials
  RU: 22 trials
  NA: 21 trials
  ES: 20 trials
  BE: 15 trials
  NO: 11 trials
  CH: 10 trials
  GB: 9 trials
  FR: 9 trials
  FI: 8 trials
  NL: 7 trials
  DACH: 6 trials
  CZ: 5 trials
  SE: 2 trials
  AU: 1 trials


In [18]:
# Resolution rate: W1-only vs touches non-W1
print("=== Resolution Rate by Localization Pattern ===\n")


# Categorize trials (only those with localization folders)
def categorize_localization(row):
    locs = row["agent_localizations"]
    if not locs:
        return None  # Will be filtered out
    if locs == {"W1"}:
        return "W1 only"
    if "W1" in locs and len(locs) > 1:
        return "W1 + other localizations"
    return "Non-W1 only"


df["loc_category"] = df.apply(categorize_localization, axis=1)

# Filter to only trials with localization folders
df_with_loc = df[df["loc_category"].notna()]

# Calculate resolution rate per category
loc_stats = (
    df_with_loc.groupby("loc_category")
    .agg(
        trials=("resolved", "count"),
        resolved=("resolved", "sum"),
    )
    .reset_index()
)
loc_stats["resolution_rate"] = (loc_stats["resolved"] / loc_stats["trials"] * 100).round(1)

print("Resolution rate by localization pattern:\n")
display(loc_stats)

# Visualize
fig = px.bar(
    loc_stats,
    x="loc_category",
    y="resolution_rate",
    text="resolution_rate",
    title="Resolution Rate by Localization Pattern",
    color="loc_category",
    color_discrete_map={
        "W1 only": "#00CC96",
        "W1 + other localizations": "#FFA15A",
        "Non-W1 only": "#EF553B",
    },
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    xaxis_title="Localization Pattern",
    yaxis_title="Resolution Rate (%)",
    yaxis_range=[0, 100],
    showlegend=False,
)
fig.show()

=== Resolution Rate by Localization Pattern ===

Resolution rate by localization pattern:



,loc_category,trials,resolved,resolution_rate
0,W1 + other localizations,50,25,50.0
1,W1 only,453,270,59.6


In [19]:
# Per-task analysis: do tasks where agent touches multiple localizations have different outcomes?
print("=== Per-Task Analysis: Multi-Localization Changes ===\n")

# Aggregate by task
task_loc_stats = (
    df.groupby("instance_id")
    .agg(
        project=("project", "first"),
        total_runs=("resolved", "count"),
        resolved_runs=("resolved", "sum"),
        ever_touched_non_w1=("touches_non_w1", "any"),
        runs_touching_non_w1=("touches_non_w1", "sum"),
        all_localizations=("agent_localizations", lambda x: set().union(*x)),
    )
    .reset_index()
)

task_loc_stats["resolution_rate"] = (task_loc_stats["resolved_runs"] / task_loc_stats["total_runs"] * 100).round(1)

# Split into tasks that ever touch non-W1 vs those that don't
tasks_with_non_w1 = task_loc_stats[task_loc_stats["ever_touched_non_w1"]]
tasks_without_non_w1 = task_loc_stats[~task_loc_stats["ever_touched_non_w1"]]

print(f"Tasks where agent EVER touched non-W1 localizations: {len(tasks_with_non_w1)}")
if len(tasks_with_non_w1) > 0:
    avg_rate = tasks_with_non_w1["resolution_rate"].mean()
    print(f"  Average resolution rate: {avg_rate:.1f}%")

print(f"\nTasks where agent NEVER touched non-W1 localizations: {len(tasks_without_non_w1)}")
if len(tasks_without_non_w1) > 0:
    avg_rate = tasks_without_non_w1["resolution_rate"].mean()
    print(f"  Average resolution rate: {avg_rate:.1f}%")

# Show details of tasks that touch multiple localizations
if len(tasks_with_non_w1) > 0:
    print("\n=== Tasks Where Agent Touched Non-W1 Localizations ===\n")
    display(tasks_with_non_w1[["instance_id", "project", "resolution_rate", "runs_touching_non_w1", "all_localizations"]].sort_values("resolution_rate"))

=== Per-Task Analysis: Multi-Localization Changes ===

Tasks where agent EVER touched non-W1 localizations: 25
  Average resolution rate: 52.0%

Tasks where agent NEVER touched non-W1 localizations: 76
  Average resolution rate: 60.5%

=== Tasks Where Agent Touched Non-W1 Localizations ===



,instance_id,project,resolution_rate,runs_touching_non_w1,all_localizations
15,microsoftInternal__NAV-183399,BaseApp,0.0,5,"{GB, APAC, IT, NO, CH, ES, FI, W1, RU}"
30,microsoftInternal__NAV-206527,BaseApp,0.0,2,"{W1, IT}"
29,microsoftInternal__NAV-206135,BaseApp,0.0,1,"{W1, RU, IT}"
37,microsoftInternal__NAV-208649,BaseApp,0.0,3,"{W1, IT}"
59,microsoftInternal__NAV-215645,SubscriptionBilling,0.0,2,"{W1, NA}"
63,microsoftInternal__NAV-216918,BaseApp,0.0,2,"{W1, NA, BE, IT}"
74,microsoftInternal__NAV-220036,BaseApp,0.0,1,"{W1, GB, NA, APAC, NO, ES}"
46,microsoftInternal__NAV-211521,BaseApp,20.0,4,"{BE, APAC, IT, FR, W1}"
28,microsoftInternal__NAV-205825,BaseApp,40.0,1,"{NA, BE, APAC, IT, ES, CZ, FR, SE, RU, GB, NO,..."
85,microsoftInternal__NAV-223819,BaseApp,40.0,1,"{GB, APAC, IT, CH, ES, FR, W1, RU}"


In [20]:
# NEW: For tasks where agent sometimes touched non-W1, compare outcomes
print("\n=== Within-Task Analysis: Non-W1 Touches vs Not ===\n")
print("For tasks where agent touched non-W1 in SOME but not ALL runs,")
print("did it resolve better when it avoided non-W1 localizations?\n")

# Get tasks where agent touched non-W1 in some but not all runs
mixed_tasks = task_loc_stats[(task_loc_stats["runs_touching_non_w1"] > 0) & (task_loc_stats["runs_touching_non_w1"] < task_loc_stats["total_runs"])]

if len(mixed_tasks) > 0:
    # For each mixed task, compare resolution when touching vs not touching non-W1
    mixed_task_ids = mixed_tasks["instance_id"].tolist()
    df_mixed = df[df["instance_id"].isin(mixed_task_ids)].copy()

    # Resolution rate when touching non-W1 vs not
    touched_resolved = df_mixed[df_mixed["touches_non_w1"]]["resolved"].sum()
    touched_total = df_mixed["touches_non_w1"].sum()
    not_touched_resolved = df_mixed[~df_mixed["touches_non_w1"]]["resolved"].sum()
    not_touched_total = len(df_mixed) - touched_total

    print(f"Tasks with mixed behavior: {len(mixed_tasks)}")
    print(f"\nRuns where agent touched non-W1: {touched_total}")
    print(f"  Resolved: {touched_resolved} ({100 * touched_resolved / touched_total:.1f}%)")
    print(f"\nRuns where agent did NOT touch non-W1: {not_touched_total}")
    print(f"  Resolved: {not_touched_resolved} ({100 * not_touched_resolved / not_touched_total:.1f}%)")

    # Per-task breakdown
    print("\n=== Per-Task Breakdown (Mixed Behavior Tasks) ===\n")
    breakdown = []
    for task_id in mixed_task_ids:
        task_df = df_mixed[df_mixed["instance_id"] == task_id]
        touched_runs = task_df[task_df["touches_non_w1"]]
        not_touched_runs = task_df[~task_df["touches_non_w1"]]
        breakdown.append(
            {
                "instance_id": task_id,
                "runs_touched_non_w1": len(touched_runs),
                "resolved_when_touched": touched_runs["resolved"].sum(),
                "runs_not_touched": len(not_touched_runs),
                "resolved_when_not_touched": not_touched_runs["resolved"].sum(),
            }
        )
    breakdown_df = pd.DataFrame(breakdown)
    breakdown_df["touch_rate"] = (breakdown_df["resolved_when_touched"] / breakdown_df["runs_touched_non_w1"] * 100).round(0).astype(int).astype(str) + "%"
    breakdown_df["no_touch_rate"] = (breakdown_df["resolved_when_not_touched"] / breakdown_df["runs_not_touched"] * 100).round(0).astype(int).astype(str) + "%"
    display(breakdown_df[["instance_id", "runs_touched_non_w1", "resolved_when_touched", "touch_rate", "runs_not_touched", "resolved_when_not_touched", "no_touch_rate"]])
else:
    print("No tasks with mixed behavior (agent either always or never touched non-W1).")


=== Within-Task Analysis: Non-W1 Touches vs Not ===

For tasks where agent touched non-W1 in SOME but not ALL runs,
did it resolve better when it avoided non-W1 localizations?

Tasks with mixed behavior: 23

Runs where agent touched non-W1: 40
  Resolved: 20 (50.0%)

Runs where agent did NOT touch non-W1: 75
  Resolved: 40 (53.3%)

=== Per-Task Breakdown (Mixed Behavior Tasks) ===



,instance_id,runs_touched_non_w1,resolved_when_touched,touch_rate,runs_not_touched,resolved_when_not_touched,no_touch_rate
0,microsoftInternal__NAV-174794,2,2,100%,3,3,100%
1,microsoftInternal__NAV-176082,1,0,0%,4,2,50%
2,microsoftInternal__NAV-176194,1,1,100%,4,2,50%
3,microsoftInternal__NAV-185696,1,1,100%,4,4,100%
4,microsoftInternal__NAV-205825,1,1,100%,4,1,25%
5,microsoftInternal__NAV-206135,1,0,0%,4,0,0%
6,microsoftInternal__NAV-206527,2,0,0%,3,0,0%
7,microsoftInternal__NAV-207236,1,0,0%,4,3,75%
8,microsoftInternal__NAV-207247,2,2,100%,3,3,100%
9,microsoftInternal__NAV-208649,3,0,0%,2,0,0%


In [21]:
# Failure category breakdown for trials that touch non-W1
print("=== Failure Analysis for Trials Touching Non-W1 ===\n")

non_w1_trials = df[df["touches_non_w1"]].copy()
if len(non_w1_trials) > 0:
    non_w1_trials["category"] = non_w1_trials["failure_category"].fillna("Unreviewed")
    non_w1_trials.loc[non_w1_trials["resolved"], "category"] = "Resolved"

    outcome_counts = non_w1_trials["category"].value_counts()
    print(f"Outcome distribution for {len(non_w1_trials)} trials touching non-W1 localizations:")
    for outcome, count in outcome_counts.items():
        print(f"  {outcome}: {count} ({100 * count / len(non_w1_trials):.1f}%)")

    # Visualize
    fig = px.pie(
        values=outcome_counts.values,
        names=outcome_counts.index,
        title=f"Outcomes for Trials Touching Non-W1 Localizations ({len(non_w1_trials)} trials)",
        color=outcome_counts.index,
        color_discrete_map={
            "Resolved": "#00CC96",
            "Wrong Solution": "#EF553B",
            "Syntax Error": "#FFA15A",
            "Incorrect File": "#FECB52",
            "Missing Using": "#19D3F3",
            "Timeout": "#636EFA",
            "Other": "#AB63FA",
            "Unreviewed": "#B6E880",
        },
        hole=0.4,
    )
    fig.update_traces(textposition="outside", textinfo="percent+label+value")
    fig.show()
else:
    print("No trials touched non-W1 localizations.")

=== Failure Analysis for Trials Touching Non-W1 ===

Outcome distribution for 50 trials touching non-W1 localizations:
  Resolved: 25 (50.0%)
  Wrong Solution: 17 (34.0%)
  Syntax Error: 6 (12.0%)
  Incorrect File: 2 (4.0%)
